In [0]:
%sql
SHOW CATALOGS


In [0]:
%sql
SHOW SCHEMAS IN workspace;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.csv_ingestion;

In [0]:
%sql
SHOW SCHEMAS IN workspace;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.csv_ingestion.raw_files;

In [0]:
%sql
SHOW VOLUMES IN workspace.csv_ingestion;

In [0]:
catalog_name = "workspace"

schemas = spark.sql(f"SHOW SCHEMAS IN {catalog_name}")

schemas.show()

In [0]:
catalog_name = "workspace"

print("""
========================================
       CSV DATA INGESTION SYSTEM
========================================

1. Use existing schema
2. Create new schema

========================================
""")

choice = input("Enter your choice (1 or 2): ")

if choice == "1":

    print("\nAvailable schemas:\n")

    schemas_df = spark.sql(f"SHOW SCHEMAS IN {catalog_name}")

    schemas = [row.databaseName for row in schemas_df.collect()]

    for i, schema in enumerate(schemas, start=1):
        print(f"{i}. {schema}")

    schema_choice = input("\nEnter schema number: ")

    schema_index = int(schema_choice) - 1

    selected_schema = schemas[schema_index]

    print("\nYou selected:", selected_schema)


elif choice == "2":

    new_schema = input("\nEnter new schema name: ")

    spark.sql(
        f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{new_schema}"
    )

    selected_schema = new_schema

    print(f"\nSchema '{new_schema}' created successfully.")

else:

    print("\nInvalid choice. Please enter 1 or 2.")

In [0]:
%sql
Show schemas;

In [0]:
table_name = input("\nEnter table name: ")

print("\nSelected schema:", selected_schema)
print("Table name:", table_name)

In [0]:
num_columns = int(input("\nHow many columns do you want? "))

print(f"\nYou selected {num_columns} columns.")

In [0]:
columns = []

data_types = {
    "1": "INT",
    "2": "BIGINT",
    "3": "DOUBLE",
    "4": "STRING",
    "5": "BOOLEAN",
    "6": "DATE",
    "7": "TIMESTAMP"
}

for i in range(num_columns):

    print(f"\n--- Column {i + 1} ---")

    column_name = input("Enter column name: ")

    print("""
Choose data type:

1. INT        → Whole numbers
2. BIGINT     → Large whole numbers
3. DOUBLE     → Decimal numbers
4. STRING     → Text
5. BOOLEAN    → True / False
6. DATE       → YYYY-MM-DD
7. TIMESTAMP  → Date + Time
""")

    type_choice = input("Enter data type number (1-7): ")

    column_type = data_types.get(type_choice)

    if column_type is None:
        print("Invalid data type. Please choose 1-7.")
        break

    columns.append((column_name, column_type))


print("\nYour table structure:")

for column_name, column_type in columns:
    print(f"{column_name} → {column_type}")

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    LongType,
    DoubleType,
    StringType,
    BooleanType,
    DateType,
    TimestampType
)

In [0]:
spark_types = {
    "INT": IntegerType(),
    "BIGINT": LongType(),
    "DOUBLE": DoubleType(),
    "STRING": StringType(),
    "BOOLEAN": BooleanType(),
    "DATE": DateType(),
    "TIMESTAMP": TimestampType()
}

In [0]:
fields = []

for column_name, column_type in columns:

    spark_data_type = spark_types[column_type]

    field = StructField(
        column_name,
        spark_data_type,
        True
    )

    fields.append(field)

In [0]:
schema = StructType(fields)

In [0]:
print("\nGenerated Spark Schema:")
schema.simpleString()

In [0]:
print(schema.treeString())

In [0]:
full_table_name = f"{catalog_name}.{selected_schema}.{table_name}"

print("Table to be created:", full_table_name)

In [0]:
column_definitions = ", ".join(
    f"{field.name} {field.dataType.simpleString().upper()}"
    for field in schema.fields
)

print(column_definitions)

In [0]:
schema

In [0]:
create_table_sql = f"""
CREATE TABLE IF NOT EXISTS {full_table_name} (
    {column_definitions}
)
USING DELTA
"""

print(create_table_sql)

In [0]:
spark.sql(create_table_sql)

print(f"\nTable '{full_table_name}' created successfully!")

In [0]:
spark.sql(f"DESCRIBE {full_table_name}").show()

In [0]:
print("""
========================================
          DATA INPUT METHOD
========================================

1. Upload existing CSV file
2. Enter data manually

========================================
""")

data_choice = input("Enter your choice (1 or 2): ")

if data_choice == "1":

    print("\nYou selected: Upload existing CSV file")

elif data_choice == "2":

    print("\nYou selected: Enter data manually")

    num_records = int(input("\nHow many records do you want to enter? "))

    print(f"\nYou will enter {num_records} records.")
else:

    print("\nInvalid choice. Please enter 1 or 2.")

In [0]:
def convert_value(value, column_type):

    if value == "":
        return None

    if column_type == "INT":
        return int(value)

    elif column_type == "BIGINT":
        return int(value)

    elif column_type == "DOUBLE":
        return float(value)

    elif column_type == "STRING":
        return value

    elif column_type == "BOOLEAN":

        value = value.strip().lower()

        if value == "true":
            return True

        elif value == "false":
            return False

        else:
            raise ValueError("Please enter True or False")

    elif column_type == "DATE":

        from datetime import datetime

        return datetime.strptime(
            value,
            "%Y-%m-%d"
        ).date()

    elif column_type == "TIMESTAMP":

        return datetime.now()

In [0]:
from datetime import datetime

data = []

for record_num in range(num_records):

    print(f"\n--- Record {record_num + 1} ---")

    row = []

    for column_name, column_type in columns:

        if column_type == "TIMESTAMP":

            value = datetime.now()

            print(f"{column_name} → {value}")

        else:

            value = input(
                f"Enter {column_name} ({column_type}): "
            )

        row.append(value)

    data.append(row)

print("\nData entered successfully!")
print(data)

In [0]:
print(data)

In [0]:
# Convert data types to match schema
converted_data = []

for row in data:
    converted_row = []
    for i, (value, (col_name, col_type)) in enumerate(zip(row, columns)):
        if value == "":
            converted_row.append(None)
        elif col_type == "INT":
            converted_row.append(int(value))
        elif col_type == "BIGINT":
            converted_row.append(int(value))
        elif col_type == "DOUBLE":
            converted_row.append(float(value))
        elif col_type == "STRING":
            converted_row.append(value)
        elif col_type == "BOOLEAN":
            value = value.strip().lower()
            if value == "true":
                converted_row.append(True)
            elif value == "false":
                converted_row.append(False)
            else:
                raise ValueError("Please enter True or False")
        elif col_type == "DATE":
            from datetime import datetime
            converted_row.append(datetime.strptime(value, "%Y-%m-%d").date())
        elif col_type == "TIMESTAMP":
            converted_row.append(value)
    converted_data.append(converted_row)

manual_df = spark.createDataFrame(
    converted_data,
    schema=schema
)

print("\nManual data as Spark DataFrame:")

manual_df.show()

print("\nDataFrame schema:")

manual_df.printSchema()

In [0]:
manual_df.write \
    .mode("append") \
    .saveAsTable(full_table_name)

print(f"\nData successfully inserted into {full_table_name}")

In [0]:
spark.sql(
    f"SELECT * FROM {full_table_name}"
).show()